# Naruto Arena RL Training

Use this notebook from Google Colab or the VS Code Colab extension with a GPU runtime. The first cell clones the training repo into the Colab VM, changes into that checkout, and runs everything from there.

In [3]:
from pathlib import Path
import os
import subprocess

REPO_URL = 'https://github.com/joaovicentedev/shinobi-arena-rl.git'
PROJECT_DIR = Path('/content/shinobi-arena-rl')

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
else:
    print('repo_exists=', PROJECT_DIR)

os.chdir(PROJECT_DIR)
subprocess.run(['git', 'pull', '--ff-only'], check=True)

print('working_dir=', Path.cwd())
print('has_pyproject=', Path('pyproject.toml').exists())
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)

working_dir= /content/shinobi-arena-rl
has_pyproject= True


CompletedProcess(args=['git', 'rev-parse', '--short', 'HEAD'], returncode=0)

## Install Dependencies

In [4]:
!python -m pip install -q uv
!uv sync --extra rl

Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 41 packages in 0.82ms
Prepared 30 packages in 45.07s                                           
Installed 30 packages in 471ms                              
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.4
 + cuda-toolkit==13.0.2
 + filelock==3.29.0
 + fsspec==2026.4.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.6.1
 + numpy==2.4.4
 + nvidia-cublas==13.1.0.3
 + nvidia-cuda-cupti==13.0.85
 + nvidia-cuda-nvrtc==13.0.88
 + nvidia-cuda-runtime==13.0.96
 + nvidia-cudnn-cu13==9.19.0.56
 + nvidia-cufft==12.0.0.61
 + nvidia-cufile==1.15.1.6
 + nvidia-curand==10.4.0.35
 + nvidia-cusolver==12.0.4.66
 + nvidia-cusparse==12.6.3.3
 + nvidia-cusparselt-cu13==0.8.0
 + nvidia-nccl-cu13==2.28.9
 + nvidia-nvjitlink==13.0.88
 + nvidia-nvshmem-cu13==3.4.5
 + nvidia-nvtx==13.0.85
 + setuptools==81.0.0
 + sympy==1.14.0
 + torch==2.11.0
 + triton==3.6.0
 + typing-extensions==4.15

## Check Runtime

In [6]:
!nvidia-smi
!uv run --extra rl python - <<'PY'
import torch
from naruto_arena.rl.observation import observation_size, ROSTER, OBSERVATION_VERSION
print('cuda_available=', torch.cuda.is_available())
print('device=', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('roster_size=', len(ROSTER))
print('observation_version=', OBSERVATION_VERSION)
print('observation_size=', observation_size())

Wed May 13 02:39:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Train From Scratch

In [7]:
MODEL_PATH = 'models/colab/naruto_actor_critic_transformer_colab.pt'

!uv run --extra rl python scripts/train_rl_pytorch.py \
  --model-arch transformer \
  --device cuda \
  --opponent heuristic \
  --episodes 500 \
  --batch-episodes 32 \
  --learning-rate 3e-4 \
  --log-interval 250 \
  --save-path {MODEL_PATH}

progress=  0.20% episode=1/500 avg_return=-1.724 win_rate=  0.0% loss=n/a elapsed=1.1s
progress= 50.00% episode=250/500 avg_return=-1.678 win_rate=  0.4% loss=n/a elapsed=130.3s
progress=100.00% episode=500/500 avg_return=-1.620 win_rate=  1.6% loss=n/a elapsed=278.7s
saved_model=models/colab/naruto_actor_critic_transformer_colab.pt


## Continue Training Against An Existing RL Model

In [ ]:
INIT_MODEL = 'models/colab/naruto_actor_critic_transformer_colab.pt'
NEXT_MODEL = 'models/colab/naruto_actor_critic_transformer_colab_vs_rl.pt'

!uv run --extra rl python scripts/train_rl_pytorch.py \
  --model-arch transformer \
  --device auto \
  --init-model-path {INIT_MODEL} \
  --opponent rl \
  --opponent-model-path {INIT_MODEL} \
  --episodes 50000 \
  --batch-episodes 32 \
  --learning-rate 1e-4 \
  --log-interval 250 \
  --save-path {NEXT_MODEL}

## Quick Evaluation

In [ ]:
!uv run --extra rl python scripts/tournament_rl.py \
  --model-path {MODEL_PATH} \
  --matches-per-pair 1 \
  --max-actions 300 \
  --report-path reports/colab_tournament.json

## Optional: Copy Checkpoints To Google Drive

In [8]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/naruto-arena-models
!cp -v models/colab/*.pt /content/drive/MyDrive/naruto-arena-models/

Mounted at /content/drive
'models/colab/naruto_actor_critic_transformer_colab.pt' -> '/content/drive/MyDrive/naruto-arena-models/naruto_actor_critic_transformer_colab.pt'
